# PathVQA with Qwen2.5-VL-3B LoRA and PA-SHE

This notebook adapts the complete PathVQA PA-SHE experiment to
`Qwen/Qwen2.5-VL-3B-Instruct`. It uses the official train, validation, and test
splits; trains only LoRA adapters; selects every PA-SHE clustering and
probability-calibration choice on validation; and evaluates the locked protocol
on test once.

Research use only. This is an uncertainty-estimation experiment, not a clinical
decision-support system.


In [ ]:
!nvidia-smi

## Dependency environment

Prepare the shared virtual environment before submitting the notebook. Do not
upgrade packages from inside an active Slurm job.

```bash
source "$HOME/dissertation_2026/.venv/bin/activate"
python -m pip install -U \
  "transformers>=4.49" "peft>=0.14" accelerate safetensors \
  datasets evaluate nltk rouge-score sentence-transformers scipy scikit-learn seaborn
```

The public model is downloaded from Hugging Face on first use. Set `HF_HOME` in
the Slurm script to a persistent project cache.


In [ ]:
# Dependencies are managed in ~/dissertation_2026/.venv before submission.
import sys
from packaging.version import Version
import transformers
import peft

print("Python:", sys.executable)
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
if Version(transformers.__version__) < Version("4.49.0"):
    raise RuntimeError(
        "Qwen2.5-VL requires transformers>=4.49. Upgrade the shared environment "
        "before submitting the Slurm job."
    )
try:
    from transformers import Qwen2_5_VLForConditionalGeneration
except ImportError as error:
    raise RuntimeError(
        "This transformers installation does not expose "
        "Qwen2_5_VLForConditionalGeneration."
    ) from error


# Dataset: Complete Official PathVQA Splits

Load the complete official **train**, **validation**, and **test** splits. Training uses only `train`; early stopping uses only `validation`; final utility and PA-SHE safety evaluation use only the held-out `test` split.


In [ ]:
from datasets import load_dataset
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

# Load every official PathVQA split. Keep the test split held out.
pathvqa = load_dataset(
    "parquet",
    data_files={
        "train": "hf://datasets/flaviagiammarino/path-vqa/data/train-*.parquet",
        "validation": "hf://datasets/flaviagiammarino/path-vqa/data/validation-*.parquet",
        "test": "hf://datasets/flaviagiammarino/path-vqa/data/test-*.parquet",
    },
)
pathvqa_train = pathvqa["train"]
pathvqa_validation = pathvqa["validation"]
pathvqa_test = pathvqa["test"]

print({
    "train": len(pathvqa_train),
    "validation": len(pathvqa_validation),
    "test": len(pathvqa_test),
    "total": sum(len(split) for split in pathvqa.values()),
})

idx = 3
print(pathvqa_test[idx].keys())
print("image resolution:", np.array(pathvqa_test[idx]["image"]).shape)
plt.figure(figsize=(5, 5))
plt.imshow(pathvqa_test[idx]["image"])
plt.title(
    f"Q: {pathvqa_test[idx]['question']}\n"
    f"A: {pathvqa_test[idx]['answer']}",
    fontsize=12,
)
plt.axis("off")
plt.show()


#Prepare Dataloader

In [ ]:
import torch
from torch.utils.data import Dataset
from PIL import Image
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from torch.utils.data import DataLoader


class PathVQADataset(Dataset):
    def __init__(self, hf_dataset):
        """
        hf_dataset: HuggingFace dataset (already loaded split)
        """

        self.dataset = hf_dataset

        self.transform = transforms.Compose([
            transforms.Resize((224, 224), interpolation=InterpolationMode.BICUBIC),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        sample = self.dataset[idx]

        # --- Image ---
        # HF dataset already provides PIL image
        raw_image = sample["image"].convert('RGB')
        img = self.transform(raw_image)

        # --- Question & Answer ---
        question = sample["question"]
        answer = sample["answer"]

        return img, question, answer

# Use the complete official splits; do not resplit or subsample.
train_data = pathvqa_train
val_data = pathvqa_validation
test_data = pathvqa_test

train_dataset = PathVQADataset(train_data)
val_dataset = PathVQADataset(val_data)
test_dataset = PathVQADataset(test_data)

print(
    f"Full official splits: train={len(train_dataset)}, "
    f"validation={len(val_dataset)}, test={len(test_dataset)}"
)


img, question, answer = train_dataset[1]
print("image resolution:", img.size())
plt.figure(figsize=(5, 5))
plt.axis("off")
plt.imshow(img.permute(1, 2, 0))
plt.title(f"Q: {question}\nA: {answer}", fontsize=12)
plt.show()


# Model architecture: Qwen2.5-VL-3B-Instruct

Qwen2.5-VL combines a vision transformer with a multimodal projector and a
decoder-only language model. Unlike the original ViT–GPT-2 implementation, the
image/question fusion is already part of the pretrained model. LoRA adapters
are applied to the language-model attention projections while the pretrained
base weights remain frozen.

- Model: https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct
- Licence: Qwen Research License (non-commercial research)
- PathVQA images are processed at a fixed 224 × 224 pixel budget so the visual
  input scale remains close to the original experiment and GPU use is bounded.


## Qwen processor and experiment configuration


In [ ]:
import gc
import json
import math
import os
import random
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import torch
from PIL import Image
from torch.utils.data import DataLoader
import torchvision.transforms.functional as TVF
from transformers import AutoProcessor

BASE_MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
QWEN_DTYPE = torch.bfloat16 if device.type == "cuda" else torch.float32

# Environment variables make Slurm tuning possible without editing the notebook.
args = SimpleNamespace(
    epochs=int(os.environ.get("QWEN_EPOCHS", "10")),
    batch_size=int(os.environ.get("QWEN_TRAIN_BATCH_SIZE", "4")),
    eval_batch_size=int(os.environ.get("QWEN_EVAL_BATCH_SIZE", "4")),
    gradient_accumulation_steps=int(os.environ.get("QWEN_GRAD_ACCUM_STEPS", "4")),
    workers=int(os.environ.get("QWEN_DATALOADER_WORKERS", "8")),
    random_seed=int(os.environ.get("QWEN_RANDOM_SEED", "42")),
    lr=float(os.environ.get("QWEN_LEARNING_RATE", "1e-4")),
    weight_decay=float(os.environ.get("QWEN_WEIGHT_DECAY", "0.01")),
    early_stopping_patience=int(os.environ.get("QWEN_EARLY_STOPPING_PATIENCE", "3")),
    max_new_tokens=int(os.environ.get("QWEN_MAX_NEW_TOKENS", "32")),
    checkpoint_dir=os.environ.get(
        "QWEN_CHECKPOINT_DIR", "checkpoints_qwen2_5_vl_3b_pathvqa_pa_she"
    ),
)
REUSE_TRAINED_CHECKPOINT = os.environ.get(
    "REUSE_TRAINED_CHECKPOINT", "1"
).strip().lower() not in {"0", "false", "no"}


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(args.random_seed)

# 224 is divisible by Qwen's 28-pixel spatial factor.
processor = AutoProcessor.from_pretrained(
    BASE_MODEL_ID,
    min_pixels=224 * 224,
    max_pixels=224 * 224,
)
tokenizer = processor.tokenizer
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print({
    "base_model": BASE_MODEL_ID,
    "device": str(device),
    "dtype": str(QWEN_DTYPE),
    "train_batch_size": args.batch_size,
    "gradient_accumulation_steps": args.gradient_accumulation_steps,
    "effective_batch_size": args.batch_size * args.gradient_accumulation_steps,
    "maximum_epochs": args.epochs,
    "checkpoint_dir": args.checkpoint_dir,
    "reuse_checkpoint": REUSE_TRAINED_CHECKPOINT,
})


## Qwen answer-only training batches


In [ ]:
def qwen_raw_collate(samples):
    images, questions, answers = zip(*samples)
    return {
        "images": list(images),
        "questions": [str(question) for question in questions],
        "answers": [str(answer) for answer in answers],
    }


def qwen_image_to_pil(image):
    if isinstance(image, Image.Image):
        return image.convert("RGB")
    if torch.is_tensor(image):
        return TVF.to_pil_image(image.detach().cpu().float().clamp(0, 1)).convert("RGB")
    return Image.fromarray(np.asarray(image)).convert("RGB")


def qwen_user_messages(question):
    return [{
        "role": "user",
        "content": [
            {"type": "image"},
            {
                "type": "text",
                "text": (
                    "Answer the pathology visual question concisely using only "
                    f"the supplied image. Question: {question}"
                ),
            },
        ],
    }]


def find_last_subsequence(sequence, pattern):
    for start in range(len(sequence) - len(pattern), -1, -1):
        if sequence[start:start + len(pattern)] == pattern:
            return start
    return -1


ASSISTANT_PREFIX_IDS = tokenizer.encode(
    "<|im_start|>assistant\n", add_special_tokens=False
)
if not ASSISTANT_PREFIX_IDS:
    raise RuntimeError("Could not encode the Qwen assistant response boundary.")


def qwen_training_batch(raw_batch):
    texts = []
    for question, answer in zip(raw_batch["questions"], raw_batch["answers"]):
        conversation = qwen_user_messages(question) + [{
            "role": "assistant",
            "content": [{"type": "text", "text": answer}],
        }]
        texts.append(processor.apply_chat_template(
            conversation,
            tokenize=False,
            add_generation_prompt=False,
        ))

    tokenizer.padding_side = "right"
    batch = processor(
        text=texts,
        images=[qwen_image_to_pil(image) for image in raw_batch["images"]],
        padding=True,
        return_tensors="pt",
    )
    labels = batch["input_ids"].clone()
    for row in range(labels.shape[0]):
        valid_length = int(batch["attention_mask"][row].sum().item())
        valid_ids = labels[row, :valid_length].tolist()
        boundary_start = find_last_subsequence(valid_ids, ASSISTANT_PREFIX_IDS)
        if boundary_start < 0:
            raise RuntimeError("Assistant boundary was not found in a Qwen training example.")
        response_start = boundary_start + len(ASSISTANT_PREFIX_IDS)
        labels[row, :response_start] = -100
        labels[row, valid_length:] = -100
    batch["labels"] = labels
    return batch


def qwen_move_batch(batch):
    moved = {}
    for key, value in batch.items():
        if not torch.is_tensor(value):
            moved[key] = value
        elif value.is_floating_point():
            moved[key] = value.to(device=device, dtype=QWEN_DTYPE, non_blocking=True)
        else:
            moved[key] = value.to(device=device, non_blocking=True)
    return moved


## Qwen2.5-VL LoRA model


In [ ]:
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from transformers import Qwen2_5_VLForConditionalGeneration


def load_qwen_base_model(for_training=False):
    load_kwargs = {
        "torch_dtype": QWEN_DTYPE,
        "low_cpu_mem_usage": True,
        "attn_implementation": os.environ.get("QWEN_ATTN_IMPLEMENTATION", "sdpa"),
    }
    base = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        BASE_MODEL_ID,
        **load_kwargs,
    )
    base.to(device)
    base.config.use_cache = not for_training
    return base


def build_qwen_lora_model():
    base = load_qwen_base_model(for_training=True)
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    )
    model = get_peft_model(base, lora_config)
    try:
        model.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant": False}
        )
    except TypeError:
        model.gradient_checkpointing_enable()
    model.enable_input_require_grads()
    model.config.use_cache = False
    model.print_trainable_parameters()
    return model


def load_qwen_lora_adapter(adapter_dir):
    base = load_qwen_base_model(for_training=False)
    loaded = PeftModel.from_pretrained(base, adapter_dir, is_trainable=False)
    loaded.to(device)
    loaded.eval()
    loaded.config.use_cache = True
    return loaded


# Model training or checkpoint reuse

The notebook saves a Qwen LoRA adapter—not a second copy of the 3B base model—at
`checkpoints_qwen2_5_vl_3b_pathvqa_pa_she/best_adapter/`. By default it reuses
that adapter. Set `REUSE_TRAINED_CHECKPOINT=0` only when intentionally
retraining. Training uses the official training split, validation loss selects
the adapter, and early stopping has a default patience of three epochs.


In [ ]:
from transformers import get_cosine_schedule_with_warmup


def qwen_epoch_loss(model, data_loader, optimizer=None, scheduler=None, epoch=None):
    training = optimizer is not None
    model.train(training)
    losses = []
    if training:
        optimizer.zero_grad(set_to_none=True)

    context = torch.enable_grad() if training else torch.inference_mode()
    with context:
        for step, raw_batch in enumerate(data_loader):
            batch = qwen_move_batch(qwen_training_batch(raw_batch))
            with torch.autocast(
                device_type=device.type,
                dtype=torch.bfloat16,
                enabled=device.type == "cuda",
            ):
                loss = model(**batch).loss
            losses.append(float(loss.detach().cpu()))

            if training:
                (loss / args.gradient_accumulation_steps).backward()
                should_step = (
                    (step + 1) % args.gradient_accumulation_steps == 0
                    or step + 1 == len(data_loader)
                )
                if should_step:
                    torch.nn.utils.clip_grad_norm_(
                        (parameter for parameter in model.parameters() if parameter.requires_grad),
                        max_norm=1.0,
                    )
                    optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad(set_to_none=True)

            if training and step % 50 == 0:
                print(
                    f"Epoch {epoch}/{args.epochs} step {step}/{len(data_loader)} "
                    f"mean loss={np.mean(losses):.5f}"
                )
    return float(np.mean(losses))


checkpoint_root = Path(args.checkpoint_dir)
adapter_dir = checkpoint_root / "best_adapter"
adapter_safetensors = adapter_dir / "adapter_model.safetensors"
adapter_binary = adapter_dir / "adapter_model.bin"
adapter_exists = adapter_safetensors.is_file() or adapter_binary.is_file()
reuse_adapter = REUSE_TRAINED_CHECKPOINT and adapter_exists
checkpoint_root.mkdir(parents=True, exist_ok=True)

train_dataloader = DataLoader(
    train_dataset,
    batch_size=args.batch_size,
    shuffle=True,
    num_workers=args.workers,
    pin_memory=device.type == "cuda",
    persistent_workers=args.workers > 0,
    collate_fn=qwen_raw_collate,
)
val_dataloader = DataLoader(
    val_dataset,
    batch_size=args.eval_batch_size,
    shuffle=False,
    num_workers=args.workers,
    pin_memory=device.type == "cuda",
    persistent_workers=args.workers > 0,
    collate_fn=qwen_raw_collate,
)
print({
    "train_examples": len(train_dataset),
    "validation_examples": len(val_dataset),
    "train_batches": len(train_dataloader),
    "validation_batches": len(val_dataloader),
})

if reuse_adapter:
    print("Reusing validation-selected Qwen adapter:", adapter_dir)
else:
    if REUSE_TRAINED_CHECKPOINT:
        print("No reusable Qwen adapter found; training from scratch.")
    else:
        print("Checkpoint reuse disabled; training from scratch.")

    training_model = build_qwen_lora_model()
    trainable_parameters = [
        parameter for parameter in training_model.parameters()
        if parameter.requires_grad
    ]
    optimizer = torch.optim.AdamW(
        trainable_parameters,
        lr=args.lr,
        weight_decay=args.weight_decay,
    )
    optimizer_steps_per_epoch = math.ceil(
        len(train_dataloader) / args.gradient_accumulation_steps
    )
    total_optimizer_steps = args.epochs * optimizer_steps_per_epoch
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=max(1, int(0.03 * total_optimizer_steps)),
        num_training_steps=total_optimizer_steps,
    )

    best_validation_loss = float("inf")
    epochs_without_improvement = 0
    history = []
    for epoch in range(1, args.epochs + 1):
        train_loss = qwen_epoch_loss(
            training_model, train_dataloader, optimizer, scheduler, epoch
        )
        validation_loss = qwen_epoch_loss(training_model, val_dataloader)
        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "validation_loss": validation_loss,
        })
        print(history[-1])

        if validation_loss < best_validation_loss - 1e-6:
            best_validation_loss = validation_loss
            epochs_without_improvement = 0
            adapter_dir.mkdir(parents=True, exist_ok=True)
            training_model.save_pretrained(adapter_dir, safe_serialization=True)
            processor.save_pretrained(adapter_dir)
            (checkpoint_root / "training_history.json").write_text(
                json.dumps(history, indent=2) + "\n", encoding="utf-8"
            )
            print("Saved new validation-selected adapter:", adapter_dir)
        else:
            epochs_without_improvement += 1
            print("Epochs without validation improvement:", epochs_without_improvement)
            if epochs_without_improvement >= args.early_stopping_patience:
                print("Early stopping triggered.")
                break

    del training_model, optimizer, scheduler
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

if adapter_safetensors.is_file():
    MODEL_CHECKPOINT_PATH = str(adapter_safetensors)
elif adapter_binary.is_file():
    MODEL_CHECKPOINT_PATH = str(adapter_binary)
else:
    raise FileNotFoundError(f"Qwen LoRA adapter was not produced in {adapter_dir}")

# The frozen validation-selected model used by every later uncertainty method.
model = load_qwen_lora_adapter(adapter_dir)
print("Loaded frozen Qwen adapter:", MODEL_CHECKPOINT_PATH)


# Validation inference: qualitative sanity check

These examples use validation only. Test remains untouched until the PA-SHE
configuration has been selected and locked.


In [ ]:
import evaluate
import matplotlib.pyplot as plt
from tqdm.auto import tqdm


def qwen_generation_inputs(images, questions):
    texts = [
        processor.apply_chat_template(
            qwen_user_messages(question),
            tokenize=False,
            add_generation_prompt=True,
        )
        for question in questions
    ]
    tokenizer.padding_side = "left"
    batch = processor(
        text=texts,
        images=[qwen_image_to_pil(image) for image in images],
        padding=True,
        return_tensors="pt",
    )
    return qwen_move_batch(batch)


@torch.inference_mode()
def qwen_generate_records(
    image_question_pairs,
    *,
    do_sample,
    max_new_tokens,
    temperature=1.0,
    top_p=1.0,
    top_k=0,
    num_return_sequences=1,
    return_logprobs=False,
):
    images = [pair[0] for pair in image_question_pairs]
    questions = [str(pair[1]) for pair in image_question_pairs]
    batch = qwen_generation_inputs(images, questions)
    input_length = batch["input_ids"].shape[1]

    generation_kwargs = {
        "max_new_tokens": int(max_new_tokens),
        "do_sample": bool(do_sample),
        "num_return_sequences": int(num_return_sequences),
        "pad_token_id": tokenizer.pad_token_id,
        "eos_token_id": tokenizer.eos_token_id,
        "use_cache": True,
    }
    if do_sample:
        generation_kwargs.update({
            "temperature": float(temperature),
            "top_p": float(top_p),
            "top_k": int(top_k),
        })
    if return_logprobs:
        generation_kwargs.update({
            "return_dict_in_generate": True,
            "output_logits": True,
        })

    generated = model.generate(**batch, **generation_kwargs)
    if return_logprobs:
        sequences = generated.sequences
        raw_step_logits = getattr(generated, "logits", None)
        if raw_step_logits is None:
            raise RuntimeError(
                "Qwen generation did not return raw logits. Use transformers>=4.49."
            )
    else:
        sequences = generated
        raw_step_logits = None

    answer_ids = sequences[:, input_length:]
    answers = processor.batch_decode(
        answer_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )
    records = []
    special_stops = {tokenizer.pad_token_id}
    eos_ids = tokenizer.eos_token_id
    if isinstance(eos_ids, (list, tuple, set)):
        special_stops.update(int(token_id) for token_id in eos_ids)
    elif eos_ids is not None:
        special_stops.add(int(eos_ids))

    for row, answer in enumerate(answers):
        sequence_logprob = float("nan")
        if return_logprobs:
            token_logprobs = []
            for step, token_id in enumerate(answer_ids[row].tolist()):
                if int(token_id) in special_stops or step >= len(raw_step_logits):
                    break
                scaled_logits = raw_step_logits[step][row].float() / float(temperature)
                log_probabilities = torch.log_softmax(scaled_logits, dim=-1)
                token_logprobs.append(float(log_probabilities[int(token_id)].item()))
            sequence_logprob = (
                float(np.sum(token_logprobs)) if token_logprobs else -50.0
            )
        records.append({
            "answer": str(answer).strip(),
            "sequence_logprob": sequence_logprob,
        })
    return records


def qwen_greedy_answer(image, question, max_new_tokens=None):
    return qwen_generate_records(
        [(image, question)],
        do_sample=False,
        max_new_tokens=max_new_tokens or args.max_new_tokens,
        return_logprobs=False,
    )[0]["answer"]


sample_indices = [2, 80, 7]
fig, axes = plt.subplots(1, len(sample_indices), figsize=(14, 5))
for axis, index in zip(axes, sample_indices):
    image, question, reference = val_dataset[index]
    prediction = qwen_greedy_answer(image, question)
    axis.imshow(image.permute(1, 2, 0).clamp(0, 1))
    axis.set_title(
        f"Q: {question}\nReference: {reference}\nQwen: {prediction}",
        fontsize=8,
    )
    axis.axis("off")
plt.tight_layout()
plt.show()


# Validation utility sanity check

This is a validation-only diagnostic. The final utility values are calculated
from the greedy predictions cached during the locked official-test PA-SHE run.


In [ ]:
import re


def normalize_vqa_metric_text(text):
    text = re.sub(r"[^a-z0-9%.\-\s]", " ", str(text).lower().strip())
    return re.sub(r"\s+", " ", text).strip() or "<empty>"


def evaluate_qwen_utility_split(dataset, batch_size):
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=args.workers,
        pin_memory=device.type == "cuda",
        persistent_workers=args.workers > 0,
        collate_fn=qwen_raw_collate,
    )
    references, predictions = [], []
    for raw_batch in tqdm(loader, desc="Qwen validation greedy generation"):
        records = qwen_generate_records(
            list(zip(raw_batch["images"], raw_batch["questions"])),
            do_sample=False,
            max_new_tokens=args.max_new_tokens,
            return_logprobs=False,
        )
        references.extend(raw_batch["answers"])
        predictions.extend(record["answer"] for record in records)
    return references, predictions


def print_utility_metrics(references, predictions):
    references = [normalize_vqa_metric_text(text) for text in references]
    predictions = [normalize_vqa_metric_text(text) for text in predictions]
    bleu = evaluate.load("bleu").compute(
        predictions=predictions,
        references=references,
        max_order=1,
    )["bleu"]
    rouge_l = evaluate.load("rouge").compute(
        predictions=predictions,
        references=references,
    )["rougeL"]
    meteor = evaluate.load("meteor").compute(
        predictions=predictions,
        references=references,
    )["meteor"]
    print({"BLEU-1": bleu, "ROUGE-L": rouge_l, "METEOR": meteor})


RUN_FULL_VALIDATION_UTILITY = os.environ.get(
    "RUN_FULL_VALIDATION_UTILITY", "0"
).strip().lower() in {"1", "true", "yes"}
if RUN_FULL_VALIDATION_UTILITY:
    validation_references, validation_predictions = evaluate_qwen_utility_split(
        val_dataset,
        batch_size=int(os.environ.get("UTILITY_EVAL_BATCH_SIZE", "8")),
    )
    print_utility_metrics(validation_references, validation_predictions)
else:
    print(
        "Skipped duplicate full-validation greedy utility generation. Set "
        "RUN_FULL_VALIDATION_UTILITY=1 to run it; PA-SHE validation sampling "
        "already stores one greedy prediction per example."
    )


# Perturbation-Aware Semantic Hallucination Entropy (PA-SHE) for PathVQA/Qwen

The validation-selected Qwen LoRA model is frozen. The remaining protocol is
the same as the GPT-2 experiment: ten sampled answers under each of four input
conditions, dataset-specific semantic clustering, validation-only selection of
clustering method/threshold, sequence-probability weighting, question-type routing, and a
single locked evaluation on the official test split.


## Experimental flow

```mermaid
flowchart TB
    A["Official validation split"] --> B["Frozen VQA sampling under four conditions"]
    B --> C1["Exact-text clustering"]
    B --> C2["SBERT and BGE cosine grids<br/>0.70, 0.80, 0.90"]
    B --> C3["RoBERTa/DeBERTa mutual-NLI grids<br/>0.35, 0.50, 0.65"]
    C1 --> D["Validation PA-SHE AUROC<br/>failure = ROUGE-L below 0.50"]
    C2 --> D
    C3 --> D
    D --> E["Lock overall and open-ended winners"]

    F["Official test split"] --> G["Frozen VQA sampling under four conditions"]
    E --> H["Apply locked clustering only"]
    G --> H
    H --> I["Overall and open-ended safety"]
    I --> J["Primary label: ROUGE-L below 0.50"]
    I --> K["Sensitivity only: 0.30 and 0.70"]
```


## 1. Configuration and reproducibility

The complete official validation split is used for clustering selection and
the complete official test split is used once for locked evaluation when
`PASHE_MAX_EXAMPLES = None`. Sample and feature caches are tied to the checkpoint,
dataset contents, generation settings, split, and candidate configuration.


In [ ]:
# Install once if needed:
# !pip install -q pandas scipy scikit-learn sentence-transformers transformers rouge-score seaborn

import gc
import os
import hashlib
import shutil
import json
from pathlib import Path
import math
import random
import re
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torchvision.transforms.functional as TVF
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from rouge_score import rouge_scorer
from scipy.spatial.distance import jensenshannon
from sklearn.metrics import average_precision_score, roc_auc_score
from tqdm.auto import tqdm

_pashe_limit = os.environ.get("PASHE_MAX_EXAMPLES", "").strip()
PASHE_MAX_EXAMPLES = int(_pashe_limit) if _pashe_limit else None
# Leave PASHE_MAX_EXAMPLES unset for reportable full-split results.
PASHE_NUM_SAMPLES = 10
PASHE_MAX_NEW_TOKENS = 16
PASHE_TEMPERATURE = 1.0
PASHE_TOP_P = 0.90
PASHE_RANDOM_SEED = 42
PASHE_PERTURBATION_VERSION = 2  # invalidates old samples after changing paraphrases/images

# Question-Aligned Semantic Nearest Neighbor Entropy (QA-SNNE).
QA_SNNE_NUM_SAMPLES = int(os.environ.get("QA_SNNE_NUM_SAMPLES", "20"))
QA_SNNE_TEMPERATURE = 1.0
QA_SNNE_TOP_K = 50
QA_SNNE_TOP_P = 0.90
QA_SNNE_BETA = 10.0
QA_SNNE_TAU = 1.0
QA_SNNE_CACHE_SCHEMA_VERSION = 1
QA_SNNE_EMBEDDING_MODEL = "pritamdeka/S-PubMedBert-MS-MARCO"
QA_SNNE_VARIANTS = {
    "Embedding": "qa_snne_embedding",
}
if QA_SNNE_NUM_SAMPLES < 2:
    raise ValueError("QA_SNNE_NUM_SAMPLES must be at least two.")

PASHE_LABEL_THRESHOLDS = [0.30, 0.50, 0.70]
PASHE_PRIMARY_LABEL_THRESHOLD = 0.50

# Predeclared method/threshold grid. Validation selects; test never does.
PASHE_SBERT_THRESHOLDS = [0.70, 0.80, 0.90]
PASHE_BGE_THRESHOLDS = [0.70, 0.80, 0.90]
PASHE_ROBERTA_NLI_THRESHOLDS = [0.35, 0.50, 0.65]
PASHE_DEBERTA_NLI_THRESHOLDS = [0.35, 0.50, 0.65]
# PA-SHE uses the sampled sequences' raw log-probabilities directly.
PASHE_SELECTION_CANDIDATES = (
    ["Exact text"]
    + [f"SBERT@{threshold:.2f}" for threshold in PASHE_SBERT_THRESHOLDS]
    + [f"BGE@{threshold:.2f}" for threshold in PASHE_BGE_THRESHOLDS]
    + [
        f"RoBERTa-NLI@{threshold:.2f}"
        for threshold in PASHE_ROBERTA_NLI_THRESHOLDS
    ]
    + [
        f"DeBERTa-NLI@{threshold:.2f}"
        for threshold in PASHE_DEBERTA_NLI_THRESHOLDS
    ]
)

PASHE_SBERT_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
PASHE_BGE_MODEL = "BAAI/bge-small-en-v1.5"
PASHE_ROBERTA_NLI_MODEL = "roberta-large-mnli"
PASHE_DEBERTA_NLI_MODEL = "microsoft/deberta-large-mnli"
PASHE_MODEL_BATCH_SIZE = 64
PASHE_CACHE_SCHEMA_VERSION = 4
PASHE_FEATURE_SCHEMA_VERSION = 7

required = [
    "model", "tokenizer", "device", "train_dataset", "val_dataset",
    "test_dataset", "MODEL_CHECKPOINT_PATH",
]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        "Run the Qwen model loading/evaluation cells first. Missing: "
        + ", ".join(missing)
    )

PASHE_CHECKPOINT_PATH = Path(MODEL_CHECKPOINT_PATH).resolve()
if not PASHE_CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"Checkpoint not found: {PASHE_CHECKPOINT_PATH}")
_pashe_cache_root = os.environ.get("PASHE_CACHE_ROOT", "").strip()
_pashe_scratch_root = Path.home() / "scratch"
if _pashe_cache_root:
    PASHE_CACHE_DIR = Path(_pashe_cache_root).expanduser().resolve()
    _pashe_cache_source = "PASHE_CACHE_ROOT environment variable"
elif _pashe_scratch_root.is_dir():
    PASHE_CACHE_DIR = (
        _pashe_scratch_root
        / "dissertation_2026"
        / "pa_she_pathvqa_qwen_cache"
    ).resolve()
    _pashe_cache_source = "automatic ~/scratch selection"
else:
    PASHE_CACHE_DIR = (
        PASHE_CHECKPOINT_PATH.parent / "pa_she_cache_pathvqa"
    )
    _pashe_cache_source = "checkpoint-adjacent fallback"
PASHE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Qwen sampling caches can be large. Fail before generation if the selected
# filesystem is nearly full instead of losing a multi-hour run during flush.
PASHE_MIN_CACHE_FREE_GB = float(
    os.environ.get("PASHE_MIN_CACHE_FREE_GB", "5.0")
)
_pashe_disk_usage = shutil.disk_usage(PASHE_CACHE_DIR)
_pashe_cache_free_gb = _pashe_disk_usage.free / (1024 ** 3)
print({
    "pa_she_cache_dir": str(PASHE_CACHE_DIR),
    "cache_location_source": _pashe_cache_source,
    "cache_free_gb": round(_pashe_cache_free_gb, 2),
    "minimum_required_free_gb": PASHE_MIN_CACHE_FREE_GB,
})
if _pashe_cache_free_gb < PASHE_MIN_CACHE_FREE_GB:
    raise OSError(
        f"PA-SHE cache filesystem has only {_pashe_cache_free_gb:.2f} GiB free; "
        f"at least {PASHE_MIN_CACHE_FREE_GB:.2f} GiB is required. "
        "Set PASHE_CACHE_ROOT to a spacious directory such as "
        "$HOME/scratch/dissertation_2026/pa_she_pathvqa_qwen_cache."
    )


## 2. Perturbations and sampled generation

Four condition families are used: original, weak image, distorted image, and question paraphrase. Paraphrases use type-preserving templates and are accepted only when question type, negation, numbers, and laterality are unchanged. If no template matches, a validated image-context wrapper is used; a rejected rewrite remains unchanged. Pathology images use optical-density H&E stain variation plus bounded scanner gamma, contrast, blur, and noise. No crop or rotation is used, so tissue geometry is preserved.

In [ ]:
def pashe_normalize(text):
    return normalize_vqa_metric_text(text)


PASHE_QUESTION_ROUTER_VERSION = 1
PASHE_CLOSED_QUESTION_PREFIXES = (
    "is " , "are " , "was " , "were " , "do " , "does " ,
    "did " , "can " , "could " , "will " , "would " ,
    "has " , "have " , "had " , "should " , "may " , "might " ,
)


def pashe_predict_question_type(question):
    normalized = re.sub(r"\s+", " ", str(question).lower().strip())
    return (
        "Closed"
        if normalized.startswith(PASHE_CLOSED_QUESTION_PREFIXES)
        else "Open"
    )


PASHE_NEGATION_TERMS = {"no", "not", "without", "absent"}
PASHE_LATERALITY_TERMS = {"left", "right", "bilateral"}


def pashe_preserved_terms(text, vocabulary):
    tokens = set(re.findall(r"[a-z0-9]+", str(text).lower()))
    return tokens.intersection(vocabulary)


def pashe_paraphrase_constraints_hold(original, candidate):
    # Do not let the perturbation change the clinical question or its answer space.
    if not str(candidate).strip():
        return False
    if pashe_predict_question_type(original) != pashe_predict_question_type(candidate):
        return False
    original_numbers = re.findall(r"\b\d+(?:\.\d+)?\b", str(original))
    candidate_numbers = re.findall(r"\b\d+(?:\.\d+)?\b", str(candidate))
    if original_numbers != candidate_numbers:
        return False
    for protected in (PASHE_NEGATION_TERMS, PASHE_LATERALITY_TERMS):
        if pashe_preserved_terms(original, protected) != pashe_preserved_terms(candidate, protected):
            return False
    return pashe_normalize(original) != pashe_normalize(candidate)


def pashe_paraphrase(question):
    q = re.sub(r"\s+", " ", str(question).strip()).rstrip("?")
    if not q:
        return str(question)
    rules = [
        (r"^what does (?:this|the) (?:image|picture) show$", "What is shown in the image?"),
        (r"^what is shown in (?:this|the) (?:image|picture)$", "What does the image show?"),
        (r"^is there (.+)$", r"Does the image show \1?"),
        (r"^does (?:this|the) image show (.+)$", r"Is \1 visible in the image?"),
        (r"^is (.+) present$", r"Does the image show \1?"),
        (r"^are there (.+)$", r"Does the image contain \1?"),
        (r"^can (.+) be seen$", r"Is \1 visible?"),
        (r"^where is (.+) located$", r"What is the location of \1?"),
        (r"^how many (.+) are (?:there|present)$", r"What number of \1 are present?"),
        (r"^what is (?:the )?diagnosis$", "Which diagnosis is most consistent with the image?"),
        (r"^what type of (.+) is (?:this|shown)$", r"Which type of \1 is shown?"),
        (r"^what is present$", "What finding is present?"),
    ]
    for pattern, replacement in rules:
        match = re.fullmatch(pattern, q, flags=re.IGNORECASE)
        if match:
            candidate = match.expand(replacement).strip()
            if pashe_paraphrase_constraints_hold(q, candidate):
                return candidate
    first_word = q.split(maxsplit=1)[0].lower()
    open_body = q[0].lower() + q[1:] if first_word in {"what", "where", "when", "why", "who", "which", "how"} else q
    fallback = (
        f"{q} according to the image?"
        if pashe_predict_question_type(q) == "Closed"
        else f"Based on the image, {open_body}?"
    )
    if pashe_paraphrase_constraints_hold(q, fallback):
        return fallback
    # Reject any unsafe rewrite instead of silently changing semantics.
    return f"{q}?"


PASHE_HE_STAIN_BASIS = torch.tensor(
    [[0.650, 0.704], [0.072, 0.990], [0.268, 0.105]], dtype=torch.float32
)
PASHE_HE_STAIN_PINV = torch.linalg.pinv(PASHE_HE_STAIN_BASIS)


def pashe_rng(seed):
    return random.Random(int(seed)) if seed is not None else random


def pashe_he_stain_shift(image, haematoxylin_scale, eosin_scale):
    x = image.detach().cpu().float().clamp(0, 1)
    if x.ndim != 3 or x.shape[0] < 3:
        return x
    rgb = x[:3].clamp(min=1.0 / 255.0)
    optical_density = -torch.log(rgb).reshape(3, -1)
    concentrations = (PASHE_HE_STAIN_PINV @ optical_density).clamp_min(0)
    base_od = PASHE_HE_STAIN_BASIS @ concentrations
    residual_od = optical_density - base_od
    scales = torch.tensor(
        [haematoxylin_scale, eosin_scale], dtype=concentrations.dtype
    ).unsqueeze(1)
    shifted_od = PASHE_HE_STAIN_BASIS @ (concentrations * scales) + residual_od
    shifted = torch.exp(-shifted_od).reshape_as(rgb)
    result = x.clone()
    result[:3] = shifted
    return result.clamp(0, 1)


def pashe_noise_like(image, standard_deviation, seed):
    generator = torch.Generator(device="cpu")
    generator.manual_seed(int(seed))
    return torch.randn(image.shape, generator=generator, dtype=image.dtype) * standard_deviation


def pashe_weak_image(image, seed=None):
    rng = pashe_rng(seed)
    x = pashe_he_stain_shift(image, rng.uniform(0.95, 1.05), rng.uniform(0.95, 1.05))
    x = TVF.adjust_gamma(x, rng.uniform(0.95, 1.05))
    x = TVF.adjust_contrast(x, rng.uniform(0.97, 1.03))
    sigma = rng.uniform(0.10, 0.35)
    return TVF.gaussian_blur(x, [3, 3], [sigma, sigma]).clamp(0, 1)


def pashe_distorted_image(image, seed=None):
    rng = pashe_rng(seed)
    x = pashe_he_stain_shift(image, rng.uniform(0.82, 1.18), rng.uniform(0.82, 1.18))
    x = TVF.adjust_gamma(x, rng.uniform(0.85, 1.15))
    x = TVF.adjust_contrast(x, rng.uniform(0.90, 1.10))
    sigma = rng.uniform(0.45, 1.10)
    x = TVF.gaussian_blur(x, [5, 5], [sigma, sigma])
    noise_seed = int(seed) + 1 if seed is not None else random.randrange(2**31)
    x = x + pashe_noise_like(x, rng.uniform(0.01, 0.035), noise_seed)
    return x.clamp(0, 1)



@torch.inference_mode()
def pashe_generate_one(image, question, do_sample=True):
    return qwen_generate_records(
        [(image, question)],
        do_sample=do_sample,
        max_new_tokens=PASHE_MAX_NEW_TOKENS,
        temperature=PASHE_TEMPERATURE,
        top_p=PASHE_TOP_P,
        top_k=0,
        return_logprobs=do_sample,
    )[0]


@torch.inference_mode()
def qa_snne_generate_samples(image, question, num_samples):
    """Generate all original-condition QA-SNNE samples in one Qwen call."""
    records = qwen_generate_records(
        [(image, question)],
        do_sample=True,
        max_new_tokens=PASHE_MAX_NEW_TOKENS,
        temperature=QA_SNNE_TEMPERATURE,
        top_p=QA_SNNE_TOP_P,
        top_k=QA_SNNE_TOP_K,
        num_return_sequences=num_samples,
        return_logprobs=False,
    )
    return [record["answer"] for record in records]


def pashe_collect_example(dataset, dataset_index, split_name):
    image, question, reference = dataset[dataset_index]
    paraphrase = pashe_paraphrase(question)
    split_offset = 0 if str(split_name).lower().startswith("val") else 10_000_000
    perturbation_seed = PASHE_RANDOM_SEED + split_offset + int(dataset_index) * 1000

    original_records = qwen_generate_records(
        [(image, question)],
        do_sample=True,
        max_new_tokens=PASHE_MAX_NEW_TOKENS,
        temperature=PASHE_TEMPERATURE,
        top_p=PASHE_TOP_P,
        top_k=0,
        num_return_sequences=PASHE_NUM_SAMPLES,
        return_logprobs=True,
    )
    weak_inputs = [
        (pashe_weak_image(image, perturbation_seed + 100 + sample_index), question)
        for sample_index in range(PASHE_NUM_SAMPLES)
    ]
    weak_records = qwen_generate_records(
        weak_inputs,
        do_sample=True,
        max_new_tokens=PASHE_MAX_NEW_TOKENS,
        temperature=PASHE_TEMPERATURE,
        top_p=PASHE_TOP_P,
        top_k=0,
        return_logprobs=True,
    )
    distorted_inputs = [
        (pashe_distorted_image(image, perturbation_seed + 200 + sample_index), question)
        for sample_index in range(PASHE_NUM_SAMPLES)
    ]
    distorted_records = qwen_generate_records(
        distorted_inputs,
        do_sample=True,
        max_new_tokens=PASHE_MAX_NEW_TOKENS,
        temperature=PASHE_TEMPERATURE,
        top_p=PASHE_TOP_P,
        top_k=0,
        return_logprobs=True,
    )
    paraphrase_records = qwen_generate_records(
        [(image, paraphrase)],
        do_sample=True,
        max_new_tokens=PASHE_MAX_NEW_TOKENS,
        temperature=PASHE_TEMPERATURE,
        top_p=PASHE_TOP_P,
        top_k=0,
        num_return_sequences=PASHE_NUM_SAMPLES,
        return_logprobs=True,
    )

    records = []
    for condition, condition_records in {
        "original": original_records,
        "weak": weak_records,
        "distorted": distorted_records,
        "paraphrase": paraphrase_records,
    }.items():
        if len(condition_records) != PASHE_NUM_SAMPLES:
            raise RuntimeError(f"Qwen returned the wrong sample count for {condition}.")
        for record in condition_records:
            record["condition"] = condition
            records.append(record)

    greedy = pashe_generate_one(image, question, do_sample=False)["answer"]
    return {
        "cache_schema_version": PASHE_CACHE_SCHEMA_VERSION,
        "split": str(split_name),
        "dataset_index": int(dataset_index),
        "question": question,
        "paraphrase": paraphrase,
        "paraphrase_changed": pashe_normalize(question) != pashe_normalize(paraphrase),
        "reference": str(reference),
        "greedy": greedy,
        "records": records,
    }


## 3. Dataset-specific clustering backends

PathVQA validation compares Exact text with SBERT/BGE cosine thresholds
`0.70`, `0.80`, and `0.90`, plus bidirectional RoBERTa/DeBERTa-NLI
thresholds `0.35`, `0.50`, and `0.65`. Each method's pairwise score matrix is
calculated once per example and reused across its thresholds. Official test
constructs only the validation-locked overall and open-ended configurations.


In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForSequenceClassification, AutoTokenizer

pashe_sbert = SentenceTransformer(PASHE_SBERT_MODEL, device=str(device))
pashe_bge = SentenceTransformer(PASHE_BGE_MODEL, device=str(device))


def pashe_load_nli(model_name):
    tokenizer_nli = AutoTokenizer.from_pretrained(model_name)
    model_nli = AutoModelForSequenceClassification.from_pretrained(
        model_name
    ).to(device).eval()
    entailment_id = next(
        (
            int(index)
            for index, label in model_nli.config.id2label.items()
            if "entail" in str(label).lower()
        ),
        2,
    )
    return tokenizer_nli, model_nli, entailment_id


pashe_roberta_tok, pashe_roberta, pashe_roberta_entail = pashe_load_nli(
    PASHE_ROBERTA_NLI_MODEL
)
pashe_deberta_tok, pashe_deberta, pashe_deberta_entail = pashe_load_nli(
    PASHE_DEBERTA_NLI_MODEL
)


def pashe_unique_answers(answers):
    normalized = [pashe_normalize(answer) for answer in answers]
    unique = list(dict.fromkeys(normalized))
    return normalized, unique


def pashe_exact_clusters(answers):
    normalized, unique = pashe_unique_answers(answers)
    mapping = {answer: index for index, answer in enumerate(unique)}
    return [mapping[answer] for answer in normalized]


def pashe_embedding_cache(answers, encoder):
    normalized, unique = pashe_unique_answers(answers)
    embeddings = encoder.encode(
        unique,
        normalize_embeddings=True,
        batch_size=PASHE_MODEL_BATCH_SIZE,
    )
    matrix = np.asarray(embeddings) @ np.asarray(embeddings).T
    return normalized, unique, matrix


@torch.inference_mode()
def pashe_nli_cache(answers, tokenizer_nli, model_nli, entailment_id):
    normalized, unique = pashe_unique_answers(answers)
    scores = np.eye(len(unique), dtype=float)
    pairs = [
        (i, j)
        for i in range(len(unique))
        for j in range(len(unique))
        if i != j
    ]
    for start in range(0, len(pairs), PASHE_MODEL_BATCH_SIZE):
        batch = pairs[start:start + PASHE_MODEL_BATCH_SIZE]
        encoded = tokenizer_nli(
            [unique[i] for i, _ in batch],
            [unique[j] for _, j in batch],
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt",
        ).to(device)
        probabilities = torch.softmax(
            model_nli(**encoded).logits,
            dim=-1,
        )[:, entailment_id]
        for (i, j), value in zip(batch, probabilities.cpu().tolist()):
            scores[i, j] = value
    return normalized, unique, scores


def pashe_clusters_from_similarity(cache, threshold, bidirectional=False):
    normalized, unique, matrix = cache
    representatives = []
    unique_cluster_ids = []
    for i in range(len(unique)):
        assigned = None
        for cluster_id, representative in enumerate(representatives):
            forward = matrix[i, representative] >= threshold
            backward = matrix[representative, i] >= threshold
            if forward and (backward if bidirectional else True):
                assigned = cluster_id
                break
        if assigned is None:
            assigned = len(representatives)
            representatives.append(i)
        unique_cluster_ids.append(assigned)
    mapping = dict(zip(unique, unique_cluster_ids))
    return [mapping[item] for item in normalized]

_qa_snne_embedding_encoder = None


def qa_snne_get_embedding_encoder():
    global _qa_snne_embedding_encoder
    if _qa_snne_embedding_encoder is None:
        _qa_snne_embedding_encoder = SentenceTransformer(
            QA_SNNE_EMBEDDING_MODEL, device=str(device)
        )
    return _qa_snne_embedding_encoder


def qa_snne_embedding_alignment(question, answers):
    encoder = qa_snne_get_embedding_encoder()
    embeddings = np.asarray(encoder.encode(
        [str(question)] + [str(answer) for answer in answers],
        normalize_embeddings=True,
        batch_size=PASHE_MODEL_BATCH_SIZE,
    ))
    return embeddings[1:] @ embeddings[0]



## 4. Risk definitions

For condition \(k\), raw sampled sequence log-probabilities are normalised
within that condition and accumulated by semantic cluster:

\[
p_k(c)=\frac{\sum_{s\in k,\ z(s)=c}\exp(\ell_s)}
{\sum_{s\in k}\exp(\ell_s)}.
\]

With \(K\) available conditions, PA-SHE uses
\(\bar p(c)=K^{-1}\sum_k p_k(c)\) and
\(H_{\mathrm{PA-SHE}}=-\sum_c\bar p(c)\log\bar p(c)\).
SE uses only \(p_{\mathrm{original}}\). Vision-Amplified Semantic Entropy is the Jensen–Shannon divergence
between weak and distorted condition distributions. SNNE uses pairwise ROUGE-L
among 20 original-input samples; QA-SNNE reweights those similarities using
question–answer embedding alignment.


In [ ]:
PASHE_CONDITIONS = ["original", "weak", "distorted", "paraphrase"]


def pashe_entropy(probabilities):
    p = np.asarray(probabilities, dtype=float)
    p = p[p > 0]
    return float(-(p * np.log(p + 1e-12)).sum())


def pashe_distribution(records, cluster_ids, condition, cluster_count):
    indices = [
        i for i, record in enumerate(records)
        if record["condition"] == condition
    ]
    distribution = np.zeros(cluster_count, dtype=float)
    if not indices:
        return distribution

    # Convert generated-answer likelihoods into within-condition sample weights.
    log_probabilities = np.asarray([
        records[i]["sequence_logprob"] for i in indices
    ], dtype=float)
    sample_weights = np.exp(log_probabilities - log_probabilities.max())
    sample_weights /= max(sample_weights.sum(), 1e-12)
    for i, sample_weight in zip(indices, sample_weights):
        distribution[int(cluster_ids[i])] += float(sample_weight)
    return distribution / max(distribution.sum(), 1e-12)

def pashe_signals(example, all_cluster_ids):
    records = example["records"]
    record_ids = all_cluster_ids
    cluster_count = max(all_cluster_ids) + 1
    distributions = {
        condition: pashe_distribution(
            records, record_ids, condition, cluster_count,
        )
        for condition in PASHE_CONDITIONS
    }
    available = [p for p in distributions.values() if p.sum() > 0]
    pooled = np.mean(available, axis=0)
    p_original = distributions["original"]
    p_weak, p_distorted = distributions["weak"], distributions["distorted"]
    vase = float(jensenshannon(
        p_weak + 1e-12, p_distorted + 1e-12, base=2.0
    ) ** 2)
    return {
        "vase": vase,
        "semantic_entropy": pashe_entropy(p_original),
        "pa_she": pashe_entropy(pooled),
        "cluster_count": int(cluster_count),
    }


pashe_rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)


def pashe_rouge_l(reference, prediction):
    reference = normalize_vqa_metric_text(reference)
    prediction = normalize_vqa_metric_text(prediction)
    return float(pashe_rouge.score(reference, prediction)["rougeL"].fmeasure)

def qa_snne_rouge_similarity_matrix(answers):
    answers = [pashe_normalize(answer) for answer in answers]
    n = len(answers)
    matrix = np.zeros((n, n), dtype=np.float64)
    for i in range(n):
        for j in range(i + 1, n):
            forward = pashe_rouge_l(answers[i], answers[j])
            backward = pashe_rouge_l(answers[j], answers[i])
            matrix[i, j] = matrix[j, i] = 0.5 * (forward + backward)
    return matrix


def qa_snne_score(similarity_matrix, alignment_scores=None):
    """Equations (1)-(4) of Carlini et al.; higher means less certain."""
    similarity = np.asarray(similarity_matrix, dtype=np.float64)
    n = similarity.shape[0]
    if similarity.shape != (n, n) or n < 2:
        raise ValueError("SNNE requires a square matrix with at least two answers.")
    if alignment_scores is not None:
        alignment = np.asarray(alignment_scores, dtype=np.float64)
        if alignment.shape != (n,):
            raise ValueError("QA-SNNE alignment scores must match sampled answers.")
        shifted = QA_SNNE_BETA * alignment
        shifted -= shifted.max()
        relevance = np.exp(shifted)
        relevance /= max(relevance.sum(), 1e-12)
        similarity = np.diag(relevance) @ similarity @ np.diag(relevance)

    row_log_sums = []
    for i in range(n):
        values = np.delete(similarity[i], i) / QA_SNNE_TAU
        maximum = float(values.max())
        row_log_sums.append(
            maximum + np.log(np.exp(values - maximum).sum() + 1e-12)
        )
    return float(-np.mean(row_log_sums))


def qa_snne_signals(example, qa_sample_example):
    answers = [str(answer) for answer in qa_sample_example["answers"]]
    if len(answers) != QA_SNNE_NUM_SAMPLES:
        raise ValueError("QA-SNNE sample count does not match configuration.")
    question = str(example["question"])
    similarity = qa_snne_rouge_similarity_matrix(answers)
    embedding_alignment = qa_snne_embedding_alignment(question, answers)
    return {
        "snne": qa_snne_score(similarity),
        "qa_snne_embedding": qa_snne_score(similarity, embedding_alignment),
        "qa_snne_embedding_alignment_mean": float(np.mean(embedding_alignment)),
    }


## 5. Validation sampling and cache

Only validation examples are sampled in this cell. The official test split is
not sampled until section 7 has selected and locked a clustering rule.


In [ ]:
PASHE_CONDITIONS = ["original", "weak", "distorted", "paraphrase"]
pashe_datasets = {
    "validation": val_dataset,
    "test": test_dataset,
}


def pashe_dataset_signature(dataset, evaluation_size):
    digest = hashlib.sha256()
    for dataset_index in range(evaluation_size):
        raw = dataset.dataset[dataset_index]
        digest.update(str(dataset_index).encode("utf-8"))
        digest.update(b"\0")
        digest.update(str(raw["question"]).encode("utf-8"))
        digest.update(b"\0")
        digest.update(str(raw["answer"]).encode("utf-8"))
        digest.update(b"\n")
    return digest.hexdigest()


def pashe_collect_split(split_name, dataset):
    evaluation_size = (
        len(dataset)
        if PASHE_MAX_EXAMPLES is None
        else min(int(PASHE_MAX_EXAMPLES), len(dataset))
    )
    checkpoint_stat = PASHE_CHECKPOINT_PATH.stat()
    cache_configuration = {
        "cache_schema_version": PASHE_CACHE_SCHEMA_VERSION,
        "split": split_name,
        "evaluation_size": evaluation_size,
        "dataset_signature": pashe_dataset_signature(dataset, evaluation_size),
        "checkpoint_path": str(PASHE_CHECKPOINT_PATH),
        "checkpoint_size": int(checkpoint_stat.st_size),
        "checkpoint_mtime_ns": int(checkpoint_stat.st_mtime_ns),
        "samples_per_condition": PASHE_NUM_SAMPLES,
        "maximum_new_tokens": PASHE_MAX_NEW_TOKENS,
        "temperature": PASHE_TEMPERATURE,
        "top_p": PASHE_TOP_P,
        "seed": PASHE_RANDOM_SEED,
        "perturbation_version": PASHE_PERTURBATION_VERSION,
        "conditions": PASHE_CONDITIONS,
        "vqa_model": BASE_MODEL_ID,
    }
    cache_hash = hashlib.sha256(
        json.dumps(cache_configuration, sort_keys=True).encode("utf-8")
    ).hexdigest()[:12]
    cache_path = PASHE_CACHE_DIR / f"{split_name}_samples_{cache_hash}.jsonl"

    cached_examples = {}
    if cache_path.exists():
        with cache_path.open("r", encoding="utf-8") as cache_file:
            for line_number, line in enumerate(cache_file, start=1):
                if not line.strip():
                    continue
                try:
                    example = json.loads(line)
                except json.JSONDecodeError:
                    print(f"Ignoring incomplete cache line {line_number}: {cache_path}")
                    continue
                dataset_index = int(example.get("dataset_index", -1))
                if example.get("cache_schema_version") != PASHE_CACHE_SCHEMA_VERSION:
                    raise ValueError("PA-SHE sample-cache schema mismatch.")
                if example.get("split") != split_name:
                    raise ValueError("PA-SHE sample-cache split mismatch.")
                if not 0 <= dataset_index < evaluation_size:
                    raise ValueError("Cached dataset index is outside this run.")
                condition_counts = {
                    condition: sum(
                        record.get("condition") == condition
                        for record in example.get("records", [])
                    )
                    for condition in PASHE_CONDITIONS
                }
                if any(
                    count != PASHE_NUM_SAMPLES
                    for count in condition_counts.values()
                ):
                    raise ValueError("Cached condition/sample counts do not match.")
                if dataset_index in cached_examples:
                    raise ValueError("Duplicate dataset index in PA-SHE cache.")
                cached_examples[dataset_index] = example

    pending_indices = [
        index for index in range(evaluation_size)
        if index not in cached_examples
    ]
    print({
        "split": split_name,
        "sample_cache": str(cache_path),
        "cached_examples": len(cached_examples),
        "pending_examples": len(pending_indices),
    })

    split_seed_offset = 0 if split_name == "validation" else 10_000_000
    with cache_path.open("a", encoding="utf-8") as cache_file:
        for dataset_index in tqdm(
            pending_indices,
            desc=f"Frozen-VQA sampling: {split_name}",
        ):
            example_seed = PASHE_RANDOM_SEED + split_seed_offset + dataset_index * 1009
            random.seed(example_seed)
            np.random.seed(example_seed)
            torch.manual_seed(example_seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(example_seed)
            example = pashe_collect_example(
                dataset=dataset,
                dataset_index=dataset_index,
                split_name=split_name,
            )
            cache_file.write(json.dumps(example, ensure_ascii=False) + "\n")
            cache_file.flush()
            cached_examples[dataset_index] = example

    return (
        [cached_examples[index] for index in range(evaluation_size)],
        cache_path,
    )


# Selection starts with validation only. Test sampling occurs in section 7,
# after PASHE_LOCKED_CLUSTERING has been assigned.
def qa_snne_collect_split(split_name, dataset):
    evaluation_size = (
        len(dataset)
        if PASHE_MAX_EXAMPLES is None
        else min(int(PASHE_MAX_EXAMPLES), len(dataset))
    )
    checkpoint_stat = PASHE_CHECKPOINT_PATH.stat()
    configuration = {
        "cache_schema_version": QA_SNNE_CACHE_SCHEMA_VERSION,
        "split": split_name,
        "evaluation_size": evaluation_size,
        "dataset_signature": pashe_dataset_signature(dataset, evaluation_size),
        "checkpoint_path": str(PASHE_CHECKPOINT_PATH),
        "checkpoint_size": int(checkpoint_stat.st_size),
        "checkpoint_mtime_ns": int(checkpoint_stat.st_mtime_ns),
        "num_samples": QA_SNNE_NUM_SAMPLES,
        "maximum_new_tokens": PASHE_MAX_NEW_TOKENS,
        "temperature": QA_SNNE_TEMPERATURE,
        "top_k": QA_SNNE_TOP_K,
        "top_p": QA_SNNE_TOP_P,
        "seed": PASHE_RANDOM_SEED,
        "input_condition": "original image and original question",
        "vqa_model": BASE_MODEL_ID,
    }
    cache_hash = hashlib.sha256(
        json.dumps(configuration, sort_keys=True).encode("utf-8")
    ).hexdigest()[:12]
    cache_path = PASHE_CACHE_DIR / f"{split_name}_qa_snne_samples_{cache_hash}.jsonl"
    cached = {}
    if cache_path.exists():
        with cache_path.open("r", encoding="utf-8") as handle:
            for line_number, line in enumerate(handle, start=1):
                if not line.strip():
                    continue
                try:
                    record = json.loads(line)
                except json.JSONDecodeError:
                    print(f"Ignoring incomplete QA-SNNE cache line {line_number}")
                    continue
                index = int(record.get("dataset_index", -1))
                if record.get("cache_schema_version") != QA_SNNE_CACHE_SCHEMA_VERSION:
                    raise ValueError("QA-SNNE sample-cache schema mismatch.")
                if record.get("split") != split_name or not 0 <= index < evaluation_size:
                    raise ValueError("QA-SNNE sample-cache split/index mismatch.")
                if len(record.get("answers", [])) != QA_SNNE_NUM_SAMPLES:
                    raise ValueError("QA-SNNE cached sample count mismatch.")
                if index in cached:
                    raise ValueError("Duplicate index in QA-SNNE sample cache.")
                cached[index] = record

    pending = [index for index in range(evaluation_size) if index not in cached]
    print({
        "split": split_name,
        "qa_snne_sample_cache": str(cache_path),
        "cached_examples": len(cached),
        "pending_examples": len(pending),
        "samples_per_example": QA_SNNE_NUM_SAMPLES,
    })
    split_offset = 30_000_000 if split_name == "validation" else 40_000_000
    with cache_path.open("a", encoding="utf-8") as handle:
        for index in tqdm(pending, desc=f"QA-SNNE sampling: {split_name}"):
            seed = PASHE_RANDOM_SEED + split_offset + index * 1013
            random.seed(seed)
            np.random.seed(seed)
            torch.manual_seed(seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(seed)
            image, question, _ = dataset[index]
            record = {
                "cache_schema_version": QA_SNNE_CACHE_SCHEMA_VERSION,
                "split": split_name,
                "dataset_index": int(index),
                "question": str(question),
                "answers": qa_snne_generate_samples(
                    image, question, QA_SNNE_NUM_SAMPLES
                ),
            }
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")
            handle.flush()
            cached[index] = record
    return [cached[index] for index in range(evaluation_size)], cache_path

pashe_validation_examples, pashe_validation_sample_cache_path = pashe_collect_split(
    "validation",
    pashe_datasets["validation"],
)
pashe_examples_by_split = {"validation": pashe_validation_examples}
pashe_sample_cache_paths = {"validation": pashe_validation_sample_cache_path}
qa_snne_validation_examples, qa_snne_validation_sample_cache_path = (
    qa_snne_collect_split("validation", pashe_datasets["validation"])
)
qa_snne_examples_by_split = {"validation": qa_snne_validation_examples}
qa_snne_sample_cache_paths = {
    "validation": qa_snne_validation_sample_cache_path
}

print({
    "validation_examples": len(pashe_validation_examples),
    "test_sampled_before_selection": False,
})
display(pd.DataFrame([{
    "split": example["split"],
    "index": example["dataset_index"],
    "question": example["question"],
    "reference": example["reference"],
    "greedy": example["greedy"],
} for example in pashe_validation_examples[:5]]))


## 6. Validation candidate features

Build validation features for the predeclared 13-candidate grid: Exact text
and three thresholds for each of SBERT, BGE, RoBERTa-NLI, and DeBERTa-NLI.
The same cache also stores Vision-Amplified Semantic Entropy, SNNE, and all QA-SNNE variants.
No test feature exists yet.


In [ ]:
def pashe_clusters_for_configurations(answers, clustering_configurations):
    clustering_configurations = list(clustering_configurations)
    unsupported = set(clustering_configurations) - set(PASHE_SELECTION_CANDIDATES)
    if unsupported:
        raise ValueError(
            "Unsupported PathVQA clustering configuration(s): "
            + ", ".join(sorted(unsupported))
        )

    requested = set(clustering_configurations)
    configurations = []
    if "Exact text" in requested:
        configurations.append(("Exact text", pashe_exact_clusters(answers)))

    embedding_specs = [
        ("SBERT", PASHE_SBERT_THRESHOLDS, pashe_sbert),
        ("BGE", PASHE_BGE_THRESHOLDS, pashe_bge),
    ]
    for method_name, thresholds, encoder in embedding_specs:
        requested_thresholds = [
            threshold for threshold in thresholds
            if f"{method_name}@{threshold:.2f}" in requested
        ]
        if requested_thresholds:
            score_cache = pashe_embedding_cache(answers, encoder)
            for threshold in requested_thresholds:
                configuration_name = f"{method_name}@{threshold:.2f}"
                configurations.append((
                    configuration_name,
                    pashe_clusters_from_similarity(
                        score_cache, threshold, bidirectional=False
                    ),
                ))

    nli_specs = [
        (
            "RoBERTa-NLI", PASHE_ROBERTA_NLI_THRESHOLDS,
            pashe_roberta_tok, pashe_roberta, pashe_roberta_entail,
        ),
        (
            "DeBERTa-NLI", PASHE_DEBERTA_NLI_THRESHOLDS,
            pashe_deberta_tok, pashe_deberta, pashe_deberta_entail,
        ),
    ]
    for method_name, thresholds, tok, mdl, entail_id in nli_specs:
        requested_thresholds = [
            threshold for threshold in thresholds
            if f"{method_name}@{threshold:.2f}" in requested
        ]
        if requested_thresholds:
            score_cache = pashe_nli_cache(answers, tok, mdl, entail_id)
            for threshold in requested_thresholds:
                configuration_name = f"{method_name}@{threshold:.2f}"
                configurations.append((
                    configuration_name,
                    pashe_clusters_from_similarity(
                        score_cache, threshold, bidirectional=True
                    ),
                ))
    if {name for name, _ in configurations} != set(clustering_configurations):
        raise ValueError("Failed to construct every requested clustering.")
    return configurations


def pashe_build_feature_frame(
    split_name,
    examples,
    sample_cache_path,
    clustering_configurations,
):
    clustering_configurations = list(clustering_configurations)
    feature_configuration = {
        "feature_schema_version": PASHE_FEATURE_SCHEMA_VERSION,
        "metric_normalization_version": 1,
        "question_router_version": PASHE_QUESTION_ROUTER_VERSION,
        "split": split_name,
        "sample_cache": sample_cache_path.name,
        "clustering_configurations": clustering_configurations,
        "sequence_weighting": "sequence-probability weighting",
        "sbert_model": PASHE_SBERT_MODEL,
        "sbert_thresholds": PASHE_SBERT_THRESHOLDS,
        "bge_model": PASHE_BGE_MODEL,
        "bge_thresholds": PASHE_BGE_THRESHOLDS,
        "roberta_nli_model": PASHE_ROBERTA_NLI_MODEL,
        "roberta_nli_thresholds": PASHE_ROBERTA_NLI_THRESHOLDS,
        "deberta_nli_model": PASHE_DEBERTA_NLI_MODEL,
        "deberta_nli_thresholds": PASHE_DEBERTA_NLI_THRESHOLDS,
        "qa_snne_sample_cache": qa_snne_sample_cache_paths[split_name].name,
        "qa_snne_num_samples": QA_SNNE_NUM_SAMPLES,
        "qa_snne_beta": QA_SNNE_BETA,
        "qa_snne_tau": QA_SNNE_TAU,
        "qa_snne_embedding_model": QA_SNNE_EMBEDDING_MODEL,
    }
    feature_hash = hashlib.sha256(
        json.dumps(feature_configuration, sort_keys=True).encode("utf-8")
    ).hexdigest()[:12]
    feature_path = PASHE_CACHE_DIR / f"{split_name}_features_{feature_hash}.csv"

    if feature_path.exists():
        frame = pd.read_csv(feature_path, keep_default_na=False)
        required_columns = {
            "split", "dataset_index", "clustering", "rougeL", "reference",
            "prediction", "answer_type", "predicted_question_type", "vase",
            "semantic_entropy", "pa_she", "cluster_count", "snne",
            "qa_snne_embedding",
        }
        missing_columns = required_columns - set(frame.columns)
        if missing_columns:
            raise ValueError(
                "Cached PathVQA features are incomplete: "
                + ", ".join(sorted(missing_columns))
            )
        expected_rows = len(examples) * len(clustering_configurations)
        if len(frame) != expected_rows:
            raise ValueError(
                f"Expected {expected_rows} cached feature rows, found {len(frame)}."
            )
        if set(frame["clustering"]) != set(clustering_configurations):
            raise ValueError("Cached PathVQA clustering set is incorrect.")
        print(f"Loaded {split_name} semantic features from {feature_path}")
        return frame, feature_path

    feature_rows = []
    for example in tqdm(examples, desc=f"Semantic clustering: {split_name}"):
        answers = [record["answer"] for record in example["records"]]
        qa_sample_example = qa_snne_examples_by_split[split_name][
            int(example["dataset_index"])
        ]
        qa_uncertainty = qa_snne_signals(example, qa_sample_example)
        configurations = pashe_clusters_for_configurations(
            answers, clustering_configurations
        )
        rouge_l = pashe_rouge_l(example["reference"], example["greedy"])
        for clustering, cluster_ids in configurations:
            row = {
                "split": split_name,
                "dataset_index": int(example["dataset_index"]),
                "clustering": clustering,
                "rougeL": rouge_l,
                "reference": example["reference"],
                "prediction": example["greedy"],
                "answer_type": ("Closed (yes/no)" if pashe_normalize(example["reference"]) in {"yes", "no"} else "Open-ended"),
                "predicted_question_type": pashe_predict_question_type(
                    example["question"]
                ),
            }
            row.update(pashe_signals(example, cluster_ids))
            row.update(qa_uncertainty)
            feature_rows.append(row)

    frame = pd.DataFrame(feature_rows)
    frame.to_csv(feature_path, index=False)
    print(f"Saved {split_name} semantic features to {feature_path}")
    return frame, feature_path


pashe_validation_features, pashe_validation_feature_cache_path = (
    pashe_build_feature_frame(
        "validation",
        pashe_validation_examples,
        pashe_validation_sample_cache_path,
        PASHE_SELECTION_CANDIDATES,
    )
)
display(pashe_validation_features.head())
print({
    "dataset": "PathVQA",
    "selection_split": "validation",
    "selection_candidates": PASHE_SELECTION_CANDIDATES,
    "sequence_weighting": "sequence-probability weighting",
    "test_features_built_before_selection": False,
})


## 7. Validation selection and locked test safety

Create two validation-only locks with PA-SHE AUROC at `ROUGE-L < 0.50`, using
AUPRC and then candidate order as tie-breakers: one lock from all validation
questions for overall/closed reporting, and one lock from open-ended validation
questions for open-ended reporting. Only after both locks are fixed are official
test features built. Test thresholds `0.30` and `0.70` are sensitivity only.


In [ ]:
PASHE_RISK_COLUMNS = {
    "Vision-Amplified Semantic Entropy": "vase",
    "SE": "semantic_entropy",
    "SNNE": "snne",
    "QA-SNNE · Embedding": "qa_snne_embedding",
    "PA-SHE": "pa_she",
}


def pashe_safe_metrics(labels, scores):
    labels = np.asarray(labels, dtype=int)
    scores = np.asarray(scores, dtype=np.float64)
    if labels.shape != scores.shape:
        raise ValueError("Labels and uncertainty scores must align.")
    if not np.isfinite(scores).all():
        raise ValueError("Uncertainty scores contain NaN or infinity.")
    if np.unique(labels).size < 2:
        return np.nan, np.nan
    return roc_auc_score(labels, scores), average_precision_score(labels, scores)


def pashe_select_clustering(validation_features, selection_subset):
    expected_examples = validation_features["dataset_index"].nunique()
    rows = []
    for candidate_order, clustering in enumerate(PASHE_SELECTION_CANDIDATES):
        group = validation_features[
            validation_features["clustering"] == clustering
        ].sort_values("dataset_index")
        if len(group) != expected_examples:
            raise ValueError(
                f"{selection_subset} validation is incomplete for {clustering}."
            )
        failures = (
            group["rougeL"].to_numpy() < PASHE_PRIMARY_LABEL_THRESHOLD
        ).astype(int)
        auroc, auprc = pashe_safe_metrics(failures, group["pa_she"])
        rows.append({
            "selection_split": "validation",
            "selection_subset": selection_subset,
            "test_used_for_selection": False,
            "candidate_order": candidate_order,
            "label_threshold": PASHE_PRIMARY_LABEL_THRESHOLD,
            "clustering": clustering,
            "sequence_weighting": "sequence-probability weighting",
            "examples": len(group),
            "failure_prevalence": failures.mean(),
            "validation_AUROC": auroc,
            "validation_AUPRC": auprc,
        })
    selection = pd.DataFrame(rows)
    ranked = selection.sort_values(
        ["validation_AUROC", "validation_AUPRC", "candidate_order"],
        ascending=[False, False, True],
        kind="mergesort",
    )
    if ranked.empty or pd.isna(ranked.iloc[0]["validation_AUROC"]):
        raise ValueError(
            f"Validation labels cannot select clustering for {selection_subset}."
        )
    return selection, str(ranked.iloc[0]["clustering"])


closed_validation_features = pashe_validation_features[
    pashe_validation_features["predicted_question_type"] == "Closed"
].copy()
if closed_validation_features.empty:
    raise ValueError("Question router predicted no closed validation questions.")
pashe_validation_selection, PASHE_LOCKED_CLUSTERING = pashe_select_clustering(
    closed_validation_features, "Predicted closed"
)

open_validation_features = pashe_validation_features[
    pashe_validation_features["predicted_question_type"] == "Open"
].copy()
if open_validation_features.empty:
    raise ValueError("PathVQA validation contains no open-ended examples.")
pashe_open_validation_selection, PASHE_OPEN_LOCKED_CLUSTERING = (
    pashe_select_clustering(open_validation_features, "Predicted open")
)

# Select the QA-SNNE alignment variant independently on validation only.
qa_validation_base = pashe_validation_features[
    pashe_validation_features["clustering"] == PASHE_SELECTION_CANDIDATES[0]
].sort_values("dataset_index")
qa_failures = (
    qa_validation_base["rougeL"].to_numpy() < PASHE_PRIMARY_LABEL_THRESHOLD
).astype(int)
qa_selection_rows = []
for variant_order, (variant, column) in enumerate(QA_SNNE_VARIANTS.items()):
    auroc, auprc = pashe_safe_metrics(qa_failures, qa_validation_base[column])
    qa_selection_rows.append({
        "selection_split": "validation",
        "selection_subset": "All",
        "test_used_for_selection": False,
        "variant_order": variant_order,
        "label_threshold": PASHE_PRIMARY_LABEL_THRESHOLD,
        "variant": variant,
        "column": column,
        "examples": len(qa_validation_base),
        "failure_prevalence": qa_failures.mean(),
        "validation_AUROC": auroc,
        "validation_AUPRC": auprc,
    })
qa_snne_validation_selection = pd.DataFrame(qa_selection_rows)
qa_ranked = qa_snne_validation_selection.sort_values(
    ["validation_AUROC", "validation_AUPRC", "variant_order"],
    ascending=[False, False, True],
    kind="mergesort",
)
if qa_ranked.empty or pd.isna(qa_ranked.iloc[0]["validation_AUROC"]):
    raise ValueError("Validation labels cannot select a QA-SNNE variant.")
QA_SNNE_LOCKED_VARIANT = str(qa_ranked.iloc[0]["variant"])
QA_SNNE_LOCKED_COLUMN = str(qa_ranked.iloc[0]["column"])
QA_SNNE_LOCKED_METHOD = f"QA-SNNE · {QA_SNNE_LOCKED_VARIANT}"

selection_path = PASHE_CACHE_DIR / "validation_selected_clustering.json"
selection_payload = {
    "dataset": "PathVQA",
    "selection_split": "validation",
    "test_used_for_selection": False,
    "question_router_version": PASHE_QUESTION_ROUTER_VERSION,
    "routing_input": "question text only; reference answer excluded",
    "primary_label_definition": "ROUGE-L < 0.50",
    "selection_metric": "AUROC; AUPRC tie-breaker; candidate order final tie-breaker",
    "eligible_candidates": PASHE_SELECTION_CANDIDATES,
    "sequence_weighting": "sequence-probability weighting",
    "closed_route_locked_clustering": PASHE_LOCKED_CLUSTERING,
    "open_route_locked_clustering": PASHE_OPEN_LOCKED_CLUSTERING,
    "qa_snne_locked_variant": QA_SNNE_LOCKED_VARIANT,
    "qa_snne_locked_column": QA_SNNE_LOCKED_COLUMN,
    "qa_snne_candidates": qa_snne_validation_selection.drop(
        columns="variant_order"
    ).to_dict(orient="records"),
    "closed_route_candidates": pashe_validation_selection.drop(
        columns="candidate_order"
    ).to_dict(orient="records"),
    "open_route_candidates": pashe_open_validation_selection.drop(
        columns="candidate_order"
    ).to_dict(orient="records"),
}
selection_path.write_text(
    json.dumps(selection_payload, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
pashe_validation_selection.to_csv(
    PASHE_CACHE_DIR / "validation_clustering_selection.csv", index=False
)
pashe_open_validation_selection.to_csv(
    PASHE_CACHE_DIR / "validation_open_ended_clustering_selection.csv", index=False
)
qa_snne_validation_selection.to_csv(
    PASHE_CACHE_DIR / "validation_qa_snne_selection.csv", index=False
)

# Test sampling starts only after both clustering routes are locked.
qa_snne_test_examples, qa_snne_test_sample_cache_path = (
    qa_snne_collect_split("test", pashe_datasets["test"])
)
qa_snne_examples_by_split["test"] = qa_snne_test_examples
qa_snne_sample_cache_paths["test"] = qa_snne_test_sample_cache_path
pashe_test_examples, pashe_test_sample_cache_path = pashe_collect_split(
    "test", pashe_datasets["test"]
)
pashe_examples_by_split["test"] = pashe_test_examples
pashe_sample_cache_paths["test"] = pashe_test_sample_cache_path
locked_clusterings = list(dict.fromkeys([
    PASHE_LOCKED_CLUSTERING, PASHE_OPEN_LOCKED_CLUSTERING,
]))
pashe_test_features, pashe_test_feature_cache_path = pashe_build_feature_frame(
    "test",
    pashe_test_examples,
    pashe_test_sample_cache_path,
    locked_clusterings,
)

pashe_locked_test_features = pashe_test_features[
    (pashe_test_features["clustering"] == PASHE_LOCKED_CLUSTERING)
    & (pashe_test_features["predicted_question_type"] == "Closed")
].sort_values("dataset_index").copy()
pashe_open_locked_test_features = pashe_test_features[
    (pashe_test_features["clustering"] == PASHE_OPEN_LOCKED_CLUSTERING)
    & (pashe_test_features["predicted_question_type"] == "Open")
].sort_values("dataset_index").copy()
if pashe_open_locked_test_features.empty:
    raise ValueError("PathVQA test contains no predicted-open examples.")
expected_test_examples = pashe_test_features["dataset_index"].nunique()
pashe_routed_test_features = pd.concat(
    [pashe_locked_test_features, pashe_open_locked_test_features],
    ignore_index=True,
).sort_values("dataset_index")
if (
    len(pashe_routed_test_features) != expected_test_examples
    or pashe_routed_test_features["dataset_index"].duplicated().any()
):
    raise ValueError("Question-type routing must cover every test example once.")

closed_validation_locked = closed_validation_features[
    closed_validation_features["clustering"] == PASHE_LOCKED_CLUSTERING
].sort_values("dataset_index")
open_validation_locked = open_validation_features[
    open_validation_features["clustering"] == PASHE_OPEN_LOCKED_CLUSTERING
].sort_values("dataset_index")


def pashe_validation_percentile(validation_scores, query_scores):
    reference = np.sort(np.asarray(validation_scores, dtype=float))
    query = np.asarray(query_scores, dtype=float)
    if reference.size == 0:
        raise ValueError("Cannot calibrate from an empty validation route.")
    return np.searchsorted(reference, query, side="right") / reference.size


for _, risk_column in PASHE_RISK_COLUMNS.items():
    calibrated_column = f"calibrated_{risk_column}"
    pashe_locked_test_features[calibrated_column] = pashe_validation_percentile(
        closed_validation_locked[risk_column], pashe_locked_test_features[risk_column]
    )
    pashe_open_locked_test_features[calibrated_column] = pashe_validation_percentile(
        open_validation_locked[risk_column],
        pashe_open_locked_test_features[risk_column],
    )
pashe_routed_test_features = pd.concat(
    [pashe_locked_test_features, pashe_open_locked_test_features],
    ignore_index=True,
).sort_values("dataset_index")
pashe_features = pashe_routed_test_features

evaluation_subsets = {
    "All": pashe_routed_test_features,
    "Closed (yes/no)": pashe_routed_test_features[
        pashe_routed_test_features["answer_type"] == "Closed (yes/no)"
    ],
    "Open-ended": pashe_routed_test_features[
        pashe_routed_test_features["answer_type"] == "Open-ended"
    ],
}
pashe_result_rows = []
for evaluation_subset, subset_features in evaluation_subsets.items():
    for label_threshold in PASHE_LABEL_THRESHOLDS:
        failures = (
            subset_features["rougeL"].to_numpy() < label_threshold
        ).astype(int)
        for method, column in PASHE_RISK_COLUMNS.items():
            score_column = f"calibrated_{column}"
            auroc, auprc = pashe_safe_metrics(failures, subset_features[score_column])
            pashe_result_rows.append({
                "evaluation_split": "official test",
                "evaluation_subset": evaluation_subset,
                "examples": len(subset_features),
                "label_threshold": label_threshold,
                "failure_prevalence": failures.mean() if len(failures) else np.nan,
                "clustering": "Question-type routed",
                "sequence_weighting": "sequence-probability weighting",
                "method": method,
                "AUROC": auroc,
                "AUPRC": auprc,
            })

pashe_results = pd.DataFrame(pashe_result_rows)
pashe_primary_results = pashe_results[
    (pashe_results["evaluation_subset"] == "All")
    & np.isclose(pashe_results["label_threshold"], PASHE_PRIMARY_LABEL_THRESHOLD)
].sort_values(["AUROC", "AUPRC"], ascending=False)
pashe_label_sensitivity = pashe_results[
    (pashe_results["evaluation_subset"] == "All")
    & (pashe_results["method"] == "PA-SHE")
].sort_values("label_threshold")
pashe_open_ended_results = pashe_results[
    pashe_results["evaluation_subset"] == "Open-ended"
].copy()
pashe_open_ended_primary_results = pashe_open_ended_results[
    np.isclose(pashe_open_ended_results["label_threshold"], PASHE_PRIMARY_LABEL_THRESHOLD)
].sort_values(["AUROC", "AUPRC"], ascending=False)
pashe_open_ended_label_sensitivity = pashe_open_ended_results[
    pashe_open_ended_results["method"] == "PA-SHE"
].sort_values("label_threshold")
pashe_closed_ended_results = pashe_results[
    pashe_results["evaluation_subset"] == "Closed (yes/no)"
].copy()

pashe_results.to_csv(PASHE_CACHE_DIR / "locked_test_safety_results.csv", index=False)
pashe_open_ended_results.to_csv(
    PASHE_CACHE_DIR / "locked_test_open_ended_safety_results.csv", index=False
)
pashe_closed_ended_results.to_csv(
    PASHE_CACHE_DIR / "locked_test_closed_ended_safety_results.csv", index=False
)

print("VALIDATION-ONLY QA-SNNE variant selection:")
display(qa_snne_validation_selection.drop(columns="variant_order").round(4))
print("Locked QA-SNNE variant before test sampling:", QA_SNNE_LOCKED_VARIANT)
print("VALIDATION-ONLY PathVQA clustering selection:")
display(pashe_validation_selection.drop(columns="candidate_order").round(4))
print("VALIDATION-ONLY open-ended clustering selection:")
display(pashe_open_validation_selection.drop(columns="candidate_order").round(4))
print("Closed-route locked clustering:", PASHE_LOCKED_CLUSTERING)
print("Open-route locked clustering:", PASHE_OPEN_LOCKED_CLUSTERING)
print("OFFICIAL TEST primary results: failure = ROUGE-L < 0.50")
display(pashe_primary_results.round(4))
print("Open-ended primary results")
display(pashe_open_ended_primary_results.round(4))
print("PA-SHE label-threshold sensitivity; clustering remains locked")
display(pashe_label_sensitivity.round(4))
print("Saved validation selection protocol:", selection_path)


## 8. Validation diagnostic and locked test comparisons

The first plot compares the overall and open-ended validation-only clustering
selections. Official-test overall results use the overall lock; open-ended
official-test results use the independently selected open-ended lock.


In [ ]:
# Validation-only diagnostics: overall and open-ended selection paths.
validation_plots = [
    ("Predicted-closed validation", pashe_validation_selection),
    ("Predicted-open validation", pashe_open_validation_selection),
]
fig, axes = plt.subplots(2, 2, figsize=(20, 10))
for row_index, (subset_label, selection_frame) in enumerate(validation_plots):
    validation_plot = selection_frame.sort_values("candidate_order")
    axes[row_index, 0].bar(
        validation_plot["clustering"],
        validation_plot["validation_AUROC"],
        color="#4c78a8",
    )
    axes[row_index, 1].bar(
        validation_plot["clustering"],
        validation_plot["validation_AUPRC"],
        color="#f58518",
    )
    axes[row_index, 0].set_title(f"{subset_label}: PA-SHE AUROC")
    axes[row_index, 1].set_title(f"{subset_label}: PA-SHE AUPRC")
    for axis in axes[row_index]:
        axis.set_ylim(0, 1)
        axis.tick_params(axis="x", rotation=45)
        axis.grid(axis="y", alpha=0.25)
fig.suptitle("PathVQA validation locks: ROUGE-L < 0.50")
plt.tight_layout()
plt.show()

# Official-test label sensitivity; clustering remains locked.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(
    pashe_label_sensitivity["label_threshold"],
    pashe_label_sensitivity["AUROC"],
    marker="o",
)
axes[1].plot(
    pashe_label_sensitivity["label_threshold"],
    pashe_label_sensitivity["AUPRC"],
    marker="o",
    color="#f58518",
)
axes[0].set_title("Official test AUROC sensitivity")
axes[1].set_title("Official test AUPRC sensitivity")
for axis in axes:
    axis.set_xlabel("ROUGE-L failure-label threshold")
    axis.set_ylim(0, 1)
    axis.set_xticks(PASHE_LABEL_THRESHOLDS)
    axis.grid(alpha=0.25)
fig.suptitle("PA-SHE with validation-locked question-type routing")
plt.tight_layout()
plt.show()

primary_comparison = pashe_primary_results.sort_values("AUROC")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].barh(primary_comparison["method"], primary_comparison["AUROC"])
axes[1].barh(
    primary_comparison["method"],
    primary_comparison["AUPRC"],
    color="#d17a22",
)
axes[0].set_title("Official test AUROC")
axes[1].set_title("Official test AUPRC")
for axis in axes:
    axis.set_xlim(0, 1)
    axis.grid(axis="x", alpha=0.25)
fig.suptitle(
    "Validation-locked question-type routing; "
    f"failure = ROUGE-L < {PASHE_PRIMARY_LABEL_THRESHOLD:.2f}"
)
plt.tight_layout()
plt.show()


## 9. Locked selective prediction and audit

These overall analyses use official-test features from the overall
validation-locked clustering configuration.


In [ ]:
primary_features = pashe_routed_test_features.copy()
rejection_fractions = np.linspace(0, 0.50, 11)
curve_rows = []

for method, column in PASHE_RISK_COLUMNS.items():
    ordered = primary_features.sort_values(
        f"calibrated_{column}", ascending=True
    )
    for fraction in rejection_fractions:
        retained_count = max(1, int(round(len(ordered) * (1.0 - fraction))))
        retained = ordered.iloc[:retained_count]
        curve_rows.append({
            "method": method,
            "rejected_fraction": fraction,
            "retained_ROUGE-L": retained["rougeL"].mean(),
        })

pashe_rejection = pd.DataFrame(curve_rows)
plt.figure(figsize=(11, 6))
for method, group in pashe_rejection.groupby("method", sort=False):
    plt.plot(
        group["rejected_fraction"],
        group["retained_ROUGE-L"],
        marker="o",
        label=method,
    )
plt.xlabel("Fraction rejected as high risk")
plt.ylabel("Mean ROUGE-L among retained answers")
plt.title("Official test selective prediction: question-type routed")
plt.grid(alpha=0.25)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

audit_columns = [
    "dataset_index", "answer_type", "reference", "prediction", "rougeL",
    "pa_she", "semantic_entropy",
]
print("Highest PA-SHE official-test cases")
display(primary_features.nlargest(10, "pa_she")[audit_columns].round(4))
print("Low PA-SHE official-test failures")
display(
    primary_features[
        primary_features["rougeL"] < PASHE_PRIMARY_LABEL_THRESHOLD
    ].nsmallest(10, "pa_she")[audit_columns].round(4)
)


## 10. Reporting checklist

- Report the 13 PathVQA candidates: Exact plus three thresholds for each of SBERT, BGE, RoBERTa-NLI, and DeBERTa-NLI.
- Report overall selection on all official validation examples at `ROUGE-L < 0.50`.
- Report open-ended selection on open-ended official validation examples only.
- Report validation AUROC selection, AUPRC tie-break, then candidate-order tie-break.
- Lock both configurations before building any official-test features.
- Use the overall lock for all/closed results and the open-ended lock only for open-ended results.
- Treat test labels `0.30` and `0.70` as sensitivity analyses, not selection.
- Report overall, closed-ended, and open-ended test AUROC/AUPRC separately.


## 11. Main comparison table

The table uses an overall clustering selected from all validation questions and
an open-ended clustering selected from open-ended validation questions. Both are
locked before test feature construction. Utility is shared because all risk
methods evaluate the same deployed greedy predictions. Closed-ended safety
uses exact normalized `yes`/`no` references; all other references are open-ended.


In [ ]:
# Final comparison using the two validation-locked evaluation paths.
import evaluate
from IPython.display import display

comparison_required = [
    "pashe_results", "pashe_open_ended_results", "pashe_routed_test_features",
    "PASHE_LOCKED_CLUSTERING", "PASHE_OPEN_LOCKED_CLUSTERING",
    "PASHE_PRIMARY_LABEL_THRESHOLD",
    "QA_SNNE_LOCKED_VARIANT", "QA_SNNE_LOCKED_METHOD",
]
comparison_missing = [
    name for name in comparison_required if name not in globals()
]
if comparison_missing:
    raise RuntimeError(
        "Run PA-SHE sections 5-7 first. Missing: "
        + ", ".join(comparison_missing)
    )

utility_source = pashe_routed_test_features.sort_values(
    "dataset_index"
).drop_duplicates("dataset_index")
references = [
    normalize_vqa_metric_text(text)
    for text in utility_source["reference"].astype(str)
]
predictions = [
    normalize_vqa_metric_text(text)
    for text in utility_source["prediction"].astype(str)
]
comparison_utility = {
    "BLEU-1": 100.0 * float(evaluate.load("bleu").compute(
        predictions=predictions, references=references, max_order=1
    )["bleu"]),
    "ROUGE-L": 100.0 * float(evaluate.load("rouge").compute(
        predictions=predictions, references=references
    )["rougeL"]),
    "METEOR": 100.0 * float(evaluate.load("meteor").compute(
        predictions=predictions, references=references
    )["meteor"]),
}

overall_safety = pashe_results[
    (pashe_results["evaluation_subset"] == "All")
    & np.isclose(pashe_results["label_threshold"], PASHE_PRIMARY_LABEL_THRESHOLD)
].copy()
closed_safety = pashe_results[
    (pashe_results["evaluation_subset"] == "Closed (yes/no)")
    & np.isclose(pashe_results["label_threshold"], PASHE_PRIMARY_LABEL_THRESHOLD)
].copy()
open_safety = pashe_open_ended_results[
    np.isclose(
        pashe_open_ended_results["label_threshold"],
        PASHE_PRIMARY_LABEL_THRESHOLD,
    )
].copy()


def comparison_safety(frame, method):
    match = frame[frame["method"] == method]
    if len(match) != 1:
        raise ValueError(f"Expected one locked result for {method}; found {len(match)}")
    row = match.iloc[0]
    return 100.0 * float(row["AUROC"]), 100.0 * float(row["AUPRC"])


locked_variant = (
    "Sequence-probability weighting; "
    f"Closed route={PASHE_LOCKED_CLUSTERING}; "
    f"Open route={PASHE_OPEN_LOCKED_CLUSTERING}"
)
specification = [
    ("Semantic entropy", locked_variant, "SE"),
    ("Semantic nearest-neighbour entropy", f"ROUGE-L · n={QA_SNNE_NUM_SAMPLES}", "SNNE"),
    ("Vision-Amplified Semantic Entropy", locked_variant, "Vision-Amplified Semantic Entropy"),
    ("Perturbation-Aware Semantic Hallucination Entropy (PA-SHE)", locked_variant, "PA-SHE"),
]
for qa_variant in QA_SNNE_VARIANTS:
    selected_suffix = (
        " · validation-selected"
        if qa_variant == QA_SNNE_LOCKED_VARIANT else ""
    )
    specification.append((
        "Question-aligned SNNE",
        f"{qa_variant}{selected_suffix} · beta={QA_SNNE_BETA:g}",
        f"QA-SNNE · {qa_variant}",
    ))

rows = []
for family, variant, method in specification:
    overall_auroc, overall_auprc = comparison_safety(overall_safety, method)
    closed_auroc, closed_auprc = comparison_safety(closed_safety, method)
    open_auroc, open_auprc = comparison_safety(open_safety, method)
    rows.append({
        "Uncertainty method": family,
        "Variant / clustering": variant,
        ("Utility", "BLEU-1"): comparison_utility["BLEU-1"],
        ("Utility", "ROUGE-L"): comparison_utility["ROUGE-L"],
        ("Utility", "METEOR"): comparison_utility["METEOR"],
        ("Overall safety", "AUROC"): overall_auroc,
        ("Overall safety", "AUPRC"): overall_auprc,
        ("Closed-ended safety", "AUROC"): closed_auroc,
        ("Closed-ended safety", "AUPRC"): closed_auprc,
        ("Open-ended safety", "AUROC"): open_auroc,
        ("Open-ended safety", "AUPRC"): open_auprc,
    })

comparison_df = pd.DataFrame(rows).set_index([
    "Uncertainty method", "Variant / clustering"
])
comparison_df.columns = pd.MultiIndex.from_tuples(
    comparison_df.columns,
    names=["Evaluation dimension", "Metric"],
)
safety_columns = [
    ("Overall safety", "AUROC"), ("Overall safety", "AUPRC"),
    ("Closed-ended safety", "AUROC"), ("Closed-ended safety", "AUPRC"),
    ("Open-ended safety", "AUROC"), ("Open-ended safety", "AUPRC"),
]
safety_maxima = {column: comparison_df[column].max() for column in safety_columns}


def highlight_maximum(value, column):
    maximum = safety_maxima.get(column)
    if maximum is not None and pd.notna(value) and np.isclose(value, maximum):
        return "font-weight: 700; background-color: #e8f1fb;"
    return ""


comparison_styler = (
    comparison_df.style
    .format("{:.2f}", na_rep="—")
    .apply(
        lambda series: [highlight_maximum(value, series.name) for value in series],
        axis=0,
    )
    .set_caption(
        "PathVQA / Qwen2.5-VL-3B: Overall, Closed-Ended, and Open-Ended Safety "
        "with Validation-Locked Routing"
    )
    .set_table_styles([
        {"selector": "caption", "props": [
            ("caption-side", "top"), ("font-size", "18px"),
            ("font-weight", "700"), ("text-align", "left"),
        ]},
        {"selector": "th", "props": [
            ("background-color", "#f5f5f5"), ("border", "1px solid #aaa"),
            ("padding", "8px"), ("text-align", "center"),
        ]},
        {"selector": "td", "props": [
            ("border", "1px solid #b5b5b5"), ("padding", "8px"),
            ("text-align", "center"),
        ]},
        {"selector": "table", "props": [
            ("border-collapse", "collapse"), ("font-size", "13px"),
            ("width", "100%"),
        ]},
    ])
)
display(comparison_styler)
comparison_df.to_csv("PathVQA_Qwen2_5_VL_3B_PA_SHE_main_comparison_table.csv")
with open(
    "PathVQA_Qwen2_5_VL_3B_PA_SHE_main_comparison_table.html",
    "w",
    encoding="utf-8",
) as comparison_file:
    comparison_file.write(comparison_styler.to_html())
print("Closed-route locked clustering:", PASHE_LOCKED_CLUSTERING)
print("Open-route locked clustering:", PASHE_OPEN_LOCKED_CLUSTERING)
print("Saved PathVQA_Qwen2_5_VL_3B_PA_SHE_main_comparison_table.csv/html")


### Reading the table

- Utility scores describe the frozen VQA answer model and repeat across risks.
- Overall safety uses all test questions; closed-ended safety uses exact normalized `yes`/`no` references; open-ended safety uses all remaining references.
- AUROC and AUPRC detect failures defined by the primary `ROUGE-L < 0.50` label.
- Semantic rows show the overall lock and the separately selected open-ended lock.
- Both locks are selected without official-test labels or scores.
- Bold cells indicate the best displayed safety ranking, not a new test selection.
